In [369]:
import pandas as pd
import numpy as np

In [370]:
df = pd.read_csv("GLD VS GC=F RollingWindow = 180.csv").fillna(0)
df

,index,GLD_Close,GLD_dailyReturns,GC=F_Close,GC=F_dailyReturns,spread,abs_sprd,GLD_rolling avg,GC=F_rolling avg,isSprdPos,...,Live Trades,GLD_PnL,GC=F_PnL,Total PnL,riskThreshold,grossCashflow,totalMTM,totalMTM in %,isStoppedOut,Adjusted PnL
0,2010-01-01,109.800003,0.000000,1117.699951,0.000000,0.000000,0.000000,0.000000,0.000000,1,...,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,0.000000
1,2010-01-02,109.800003,0.000000,1117.699951,0.000000,0.000000,0.000000,0.000000,0.000000,1,...,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,0.000000
2,2010-01-03,109.800003,0.000000,1117.699951,0.000000,0.000000,0.000000,0.000000,0.000000,1,...,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,0.000000
3,2010-01-04,109.800003,0.000000,1117.699951,0.000000,0.000000,0.000000,0.000000,0.000000,1,...,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,0.000000
4,2010-01-05,109.699997,-0.000911,1118.099976,0.000358,-0.001269,0.001269,0.000000,0.000000,0,...,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5590,2025-04-22,311.109985,-0.014196,3400.800049,-0.001585,-0.012610,0.012610,0.009315,0.009562,0,...,Opened,-35155.428345,34008.000488,0.000000,0.009562,69163.428833,0.000000,0.000000,0,0.000000
5591,2025-04-23,303.649994,-0.023979,3276.300049,-0.036609,0.012630,0.012630,0.009501,0.009969,1,...,Closed,34312.449310,-32763.000488,402.020966,0.009969,67075.449799,402.020966,0.005994,0,402.020966
5592,2025-04-24,308.070007,0.014556,3332.000000,0.017001,-0.002445,0.002445,0.009554,0.010039,0,...,0,0.000000,0.000000,0.000000,0.010039,0.000000,0.000000,0.000000,0,0.000000
5593,2025-04-25,303.095093,-0.016149,3293.600098,-0.011525,-0.004624,0.004624,0.009640,0.010083,0,...,0,0.000000,0.000000,0.000000,0.010083,0.000000,0.000000,0.000000,0,0.000000


In [371]:
close_columns = [col for col in df.columns if "_Close" in col]
left_of_close = [col.split("_Close")[0] for col in close_columns]
ticker1 = left_of_close[0]
ticker2 = left_of_close[1]

In [372]:
columns_to_filter = [
    "index",
    ticker1+"_Close", ticker1+"_dailyReturns", ticker2+"_Close", 
    ticker2+"_dailyReturns", "spread", "abs_sprd", 
    ticker1+"_rolling avg", ticker2+"_rolling avg", 
    "Live Trades", "totalMTM in %"
]

filtered_df = df[columns_to_filter]
filtered_df

,index,GLD_Close,GLD_dailyReturns,GC=F_Close,GC=F_dailyReturns,spread,abs_sprd,GLD_rolling avg,GC=F_rolling avg,Live Trades,totalMTM in %
0,2010-01-01,109.800003,0.000000,1117.699951,0.000000,0.000000,0.000000,0.000000,0.000000,0,0.000000
1,2010-01-02,109.800003,0.000000,1117.699951,0.000000,0.000000,0.000000,0.000000,0.000000,0,0.000000
2,2010-01-03,109.800003,0.000000,1117.699951,0.000000,0.000000,0.000000,0.000000,0.000000,0,0.000000
3,2010-01-04,109.800003,0.000000,1117.699951,0.000000,0.000000,0.000000,0.000000,0.000000,0,0.000000
4,2010-01-05,109.699997,-0.000911,1118.099976,0.000358,-0.001269,0.001269,0.000000,0.000000,0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...
5590,2025-04-22,311.109985,-0.014196,3400.800049,-0.001585,-0.012610,0.012610,0.009315,0.009562,Opened,0.000000
5591,2025-04-23,303.649994,-0.023979,3276.300049,-0.036609,0.012630,0.012630,0.009501,0.009969,Closed,0.005994
5592,2025-04-24,308.070007,0.014556,3332.000000,0.017001,-0.002445,0.002445,0.009554,0.010039,0,0.000000
5593,2025-04-25,303.095093,-0.016149,3293.600098,-0.011525,-0.004624,0.004624,0.009640,0.010083,0,0.000000


In [373]:
filtered_df["pos"] = np.where((filtered_df["Live Trades"] == "Closed") & (filtered_df["totalMTM in %"] > 0), int(1), 0)
filtered_df["pos"] = np.where((filtered_df["Live Trades"] == "Closed") & (filtered_df["totalMTM in %"] <= 0), int(-1), filtered_df["pos"])

filtered_df["pos"].value_counts()

pos
 0    5431
 1     147
-1      17
Name: count, dtype: int64

In [374]:
filtered_df["optimal entry"] = 0  # Initialize the column with default values

for i in range(len(filtered_df) - 1, -1, -1):  # Start from the last valid index
    if filtered_df.at[i, "pos"] == 1:
        # print(f"Processing pos=1 at index {i}")
        while i >= 0 and filtered_df.at[i, "Live Trades"] != "Opened":  # Ensure index is valid
            i -= 1
        if i >= 0:  # Check if a valid "Opened" trade was found
            # print(f"Assigning optimal entry=1 at index {i}")
            filtered_df.at[i, "optimal entry"] = 1

    elif filtered_df.at[i, "pos"] == -1:
        # print(f"Processing pos=-1 at index {i}")
        while i >= 0 and filtered_df.at[i, "Live Trades"] != "Opened":  # Ensure index is valid
            i -= 1
        if i >= 0:  # Check if a valid "Opened" trade was found
            # print(f"Assigning optimal entry=-1 at index {i}")
            filtered_df.at[i, "optimal entry"] = -1

# Check the counts
print("Counts in 'optimal entry':")
print(filtered_df["optimal entry"].value_counts())

Counts in 'optimal entry':
optimal entry
 0    5431
 1     147
-1      17
Name: count, dtype: int64


In [375]:
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam


# Prepare features and labels
X = filtered_df[[ticker1+"_dailyReturns", 
    ticker2+"_dailyReturns", "abs_sprd", ticker1+"_rolling avg", ticker2+"_rolling avg"]]

y = (filtered_df["optimal entry"])

# Convert labels to categorical (one-hot encoding)
y_categorical = to_categorical(y + 1)  # Shift labels (-1, 0, 1) to (0, 1, 2)

# Normalize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_categorical, test_size=0.2, random_state=42)

# Define the model
model = tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(3, activation='softmax')
])

optimizer = Adam(learning_rate=0.00005)

# Compile the model
model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()])

from tensorflow.keras.callbacks import EarlyStopping
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

model.fit(X_train, y_train, epochs=1000, batch_size=32, validation_data=(X_test, y_test), callbacks=[early_stopping])
# model.fit(X_train, y_train, epochs=100, batch_size=32, validation_data=(X_test, y_test))

Epoch 1/1000
140/140 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.7522 - loss: 0.9428 - precision_29: 0.8960 - recall_29: 0.1075 - val_accuracy: 0.9651 - val_loss: 0.8261 - val_precision_29: 0.8889 - val_recall_29: 0.2288
Epoch 2/1000
140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9724 - loss: 0.7733 - precision_29: 0.9220 - recall_29: 0.2800 - val_accuracy: 0.9660 - val_loss: 0.6878 - val_precision_29: 0.9356 - val_recall_29: 0.4933
Epoch 3/1000
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9747 - loss: 0.6391 - precision_29: 0.9566 - recall_29: 0.5550 - val_accuracy: 0.9660 - val_loss: 0.5766 - val_precision_29: 0.9560 - val_recall_29: 0.7382
Epoch 4/1000
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9734 - loss: 0.5299 - precision_29: 0.9691 - recall_29: 0.8336 - val_accuracy: 0.9660 - val_loss: 0.4842 - val_precision_29: 0.9649 - val_recall_29: 0.9339
Epoch 5/1000
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9695 - loss: 0.4471 - precision_29: 

In [377]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# predictions = model.predict(X_test)

# # Convert probabilities to class labels
# predicted_classes = np.argmax(predictions, axis=1) - 1  # Shift back to original labels (-1, 0, 1)
# y_test_classes = np.argmax(y_test, axis=1) - 1

predictions = model.predict(X_scaled)

# Convert probabilities to class labels
predicted_classes = np.argmax(predictions, axis=1) - 1

y_test_classes = y

# Define the labels
labels = [-1, 0, 1]

# Calculate metrics for each label
for label in labels:
    # Accuracy for the specific label
    label_accuracy = accuracy_score(y_test_classes == label, predicted_classes == label)
    print(f"Accuracy for label {label}: {label_accuracy:.2f}")
    
    # Precision for the specific label
    label_precision = precision_score(y_test_classes, predicted_classes, labels=[label], average='macro', zero_division=0)
    print(f"Precision for label {label}: {label_precision:.2f}")
    
    # Recall for the specific label
    label_recall = recall_score(y_test_classes, predicted_classes, labels=[label], average='macro', zero_division=0)
    print(f"Recall for label {label}: {label_recall:.2f}")
    
    # F1-score for the specific label
    label_f1 = f1_score(y_test_classes, predicted_classes, labels=[label], average='macro', zero_division=0)
    print(f"F1-Score for label {label}: {label_f1:.2f}")
    
    print("-" * 30)  # Separator for readability

175/175 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
Accuracy for label -1: 1.00
Precision for label -1: 0.00
Recall for label -1: 0.00
F1-Score for label -1: 0.00
------------------------------
Accuracy for label 0: 0.98
Precision for label 0: 0.99
Recall for label 0: 0.99
F1-Score for label 0: 0.99
------------------------------
Accuracy for label 1: 0.98
Precision for label 1: 0.71
Recall for label 1: 0.67
F1-Score for label 1: 0.69
------------------------------


In [395]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense
from sklearn.preprocessing import StandardScaler
import random

X = filtered_df[[ticker1+"_dailyReturns", 
    ticker2+"_dailyReturns","abs_sprd", ticker1+"_rolling avg", ticker2+"_rolling avg"]]

y = (filtered_df["optimal entry"])

# Normalize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Define RL environment parameters
n_actions = 3  # Actions: -1, 0, 1
n_states = X_scaled.shape[1]  # Number of features
gamma = 0.95  # Discount factor
epsilon = 1.0  # Exploration rate
epsilon_min = 0.01
epsilon_decay = 0.995
learning_rate = 0.001

# Define the Q-network
def build_q_network():
    model = Sequential([
        Dense(64, activation='relu', input_shape=(n_states,)),
        Dense(32, activation='relu'),
        Dense(n_actions, activation='linear')  # Output Q-values for each action
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate), loss='mse')
    return model

# Initialize Q-network and target network
q_network = build_q_network()
target_network = build_q_network()
target_network.set_weights(q_network.get_weights())  # Synchronize weights

# Replay buffer for experience replay
replay_buffer = []
buffer_size = 10000
batch_size = 32

# Function to select an action using epsilon-greedy policy
def select_action(state, epsilon):
    if np.random.rand() < epsilon:
        return np.random.choice(n_actions)  # Explore
    q_values = q_network.predict(state[np.newaxis, :], verbose=0)
    return np.argmax(q_values)  # Exploit

# print(X_scaled.shape[1])
# Training loop
n_episodes = 1000
for episode in range(n_episodes):

    current_index = np.random.randint(0, len(X_scaled) - 1)  # Random initial index
    state = X_scaled[current_index]  # Initial state

    done = False
    total_reward = 0

    while not done:
        # Select action
        action = select_action(state, epsilon)

        # Map action to label (-1, 0, 1)
        predicted_label = action - 1

        # Calculate reward (e.g., based on correct prediction)
        true_label = y[current_index]  # Get the true label for the current index
        
        if true_label == 1 and predicted_label == true_label: # more rewards for predicting correct entry
            total_reward += 9
        elif true_label == -1 and predicted_label == true_label:
            total_reward += 9
        elif true_label == 0 and predicted_label == true_label:
            total_reward += 0
        else:
            total_reward -= 1

        # Transition to the next state
        if current_index + 1 < len(X_scaled):
            next_state = X_scaled[current_index + 1]  # Move to the next row
            current_index += 1  # Update the current index
        else:
            next_state = np.zeros_like(state)  # Define a terminal state (e.g., all zeros)
            done = True  # End the episode if at the last index
            
        # Store experience in replay buffer
        replay_buffer.append((state, action, total_reward, next_state))
        # print(replay_buffer)
        if len(replay_buffer) > buffer_size:
            replay_buffer.pop(0)

        # Sample a batch from the replay buffer
        if len(replay_buffer) >= batch_size:
            batch = random.sample(replay_buffer, batch_size)
            states, actions, rewards, next_states = zip(*batch)

            # Convert to numpy arrays
            states = np.array(states)
            actions = np.array(actions)
            rewards = np.array(rewards)
            next_states = np.array(next_states)

            # Compute target Q-values
            next_q_values = target_network.predict(next_states, verbose=0)
            max_next_q_values = np.max(next_q_values, axis=1)
            target_q_values = rewards + gamma * max_next_q_values

            # Update Q-network
            q_values = q_network.predict(states, verbose=0)
            for i, action in enumerate(actions):
                q_values[i, action] = target_q_values[i]
            q_network.fit(states, q_values, epochs=1, verbose=0, batch_size=batch_size)

        # Update state
        state = next_state

        # Check if episode is done (e.g., after a fixed number of steps)
        if total_reward >= 10 or total_reward <= -10:  # Example condition
            done = True

    # Decay epsilon
    if epsilon > epsilon_min:
        epsilon *= epsilon_decay

    # Update target network periodically
    if episode % 10 == 0:
        target_network.set_weights(q_network.get_weights())

    print(f"Episode {episode + 1}/{n_episodes}, Total Reward: {total_reward}, Epsilon: {epsilon:.2f}")

# Save the trained Q-network
q_network.save("q_network_model.keras")

Episode 1/1000, Total Reward: -10, Epsilon: 0.99
Episode 2/1000, Total Reward: -10, Epsilon: 0.99
Episode 3/1000, Total Reward: -10, Epsilon: 0.99
Episode 4/1000, Total Reward: -10, Epsilon: 0.98
Episode 5/1000, Total Reward: -10, Epsilon: 0.98
Episode 6/1000, Total Reward: -10, Epsilon: 0.97
Episode 7/1000, Total Reward: -10, Epsilon: 0.97
Episode 8/1000, Total Reward: -10, Epsilon: 0.96
Episode 9/1000, Total Reward: -10, Epsilon: 0.96
Episode 10/1000, Total Reward: -10, Epsilon: 0.95
Episode 11/1000, Total Reward: -10, Epsilon: 0.95
Episode 12/1000, Total Reward: -10, Epsilon: 0.94
Episode 13/1000, Total Reward: -10, Epsilon: 0.94
Episode 14/1000, Total Reward: -10, Epsilon: 0.93
Episode 15/1000, Total Reward: -10, Epsilon: 0.93
Episode 16/1000, Total Reward: -10, Epsilon: 0.92
Episode 17/1000, Total Reward: -10, Epsilon: 0.92
Episode 18/1000, Total Reward: -10, Epsilon: 0.91
Episode 19/1000, Total Reward: -10, Epsilon: 0.91
Episode 20/1000, Total Reward: -10, Epsilon: 0.90
Episode 2

In [397]:
from tensorflow.keras.models import load_model
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# Load the saved Q-network model
q_network = load_model("q_network_model.keras")

# Predict Q-values for the entire dataset
q_values = q_network.predict(X_scaled, verbose=0)

# Select the action with the highest Q-value for each state
predicted_actions = np.argmax(q_values, axis=1)  # Actions: 0, 1, 2

# Map actions back to labels (-1, 0, 1)
predicted_classes = predicted_actions - 1  # Shift actions to match labels

y_test_classes = y # Already in the format (-1, 0, 1)

# Define the labels
labels = [-1, 0, 1]

# Calculate metrics for each label
for label in labels:
    # Accuracy for the specific label
    label_accuracy = accuracy_score(y_test_classes == label, predicted_classes == label)
    print(f"Accuracy for label {label}: {label_accuracy:.2f}")
    
    # Precision for the specific label
    label_precision = precision_score(y_test_classes, predicted_classes, labels=[label], average='macro', zero_division=0)
    print(f"Precision for label {label}: {label_precision:.2f}")
    
    # Recall for the specific label
    label_recall = recall_score(y_test_classes, predicted_classes, labels=[label], average='macro', zero_division=0)
    print(f"Recall for label {label}: {label_recall:.2f}")
    
    # F1-score for the specific label
    label_f1 = f1_score(y_test_classes, predicted_classes, labels=[label], average='macro', zero_division=0)
    print(f"F1-Score for label {label}: {label_f1:.2f}")
    
    print("-" * 30)  # Separator for readability


Accuracy for label -1: 0.86
Precision for label -1: 0.01
Recall for label -1: 0.35
F1-Score for label -1: 0.01
------------------------------
Accuracy for label 0: 0.83
Precision for label 0: 1.00
Recall for label 0: 0.82
F1-Score for label 0: 0.90
------------------------------
Accuracy for label 1: 0.96
Precision for label 1: 0.38
Recall for label 1: 0.80
F1-Score for label 1: 0.52
------------------------------


In [398]:
predictions = q_network.predict(X_scaled, verbose=0)
# predictions = model.predict(X_scaled)

# Convert probabilities to class labels
predicted_classes = np.argmax(predictions, axis=1) - 1
df["optimal"] = filtered_df["optimal entry"]
df["predicted"] = predicted_classes
df["predicted"].replace(-1, 0,inplace=True)

In [399]:
pred = df["predicted"].tolist()
i=0

while i < len(pred):
    if pred[i] == 1:
        while i+1 < len(pred) and pred[i+1] == 1:
            pred[i+1] = 0
            i += 1
    i+=1
    
df["predicted"] = pred

In [400]:
df[["axe","optimal", "predicted"]].to_csv("NN.csv", index=False)
df["predicted"].value_counts()
# pd.read_csv("NN.csv")

predicted
0    5370
1     225
Name: count, dtype: int64

In [401]:
def signal(df,ticker1,ticker2,rollingWindow):
    
    # df.reset_index(inplace=True)
    
    df["isSprdPos"] = np.where(df["spread"] >= 0, 1, 0)
    df["signal"] = df["predicted"] 
    print(df["signal"].value_counts())
    # based on rolling window; to be accounted in param
    i = rollingWindow
    
    while i < len(df.index):
        
        absSprd = df.loc[i, "abs_sprd"]
        stock1_rolling_avg = df.loc[i, ticker1+"_rolling avg"]
        stock2_rolling_avg = df.loc[i, ticker2+"_rolling avg"]
        
        stock1_dailyReturns = df.loc[i, ticker1+"_dailyReturns"]
        stock2_dailyReturns = df.loc[i, ticker2+"_dailyReturns"]

        isNotZero = (stock1_dailyReturns != 0) & (stock2_dailyReturns != 0)
        # isNotZero = 1
        # print(df["signal"].dtype)
        if df.loc[i, "signal"]==1 and absSprd > stock1_rolling_avg and absSprd > stock2_rolling_avg and isNotZero:
            # df.loc[i, "signal"] = 1
            currSprdDir = df.loc[i, "isSprdPos"]
            # print(i, currSprdDir)
            
            i += 1
            
            while i < len(df.index) and currSprdDir == df.loc[i, "isSprdPos"]:
                df.loc[i, "signal"] = 1
                i += 1
            
            if i < len(df.index):
                df.loc[i, "signal"] = 0      
        
        i += 1
    
    df["axe"] = df["signal"].diff()
    
    return df

In [402]:
df_new = signal(df, ticker1, ticker2, rollingWindow=180)

signal
0    5370
1     225
Name: count, dtype: int64


In [403]:
df_new["axe"].value_counts()

axe
 0.0    5164
 1.0     215
-1.0     215
Name: count, dtype: int64

In [404]:
import Mean_Reversion

In [405]:
df = Mean_Reversion.pnl(df_new, 10, ticker1, ticker2, 180)
df = Mean_Reversion.pnl2(df, ticker1, ticker2)
df = Mean_Reversion.riskStrategy(df, ticker1, ticker2)
Mean_Reversion.printStats(df, ticker1, ticker2, 180)
file_path = ticker1+" VS "+ticker2+"NN.csv"
df.to_csv(ticker1+" VS "+ticker2+"NN.csv", index=False)

res = Mean_Reversion.metrics_calcs.get_sharpe(file_path)

print(f"Cumulative Returns: {res['returns']}")
print(f"Sharpe Ratio: {res['sharpe_ratio']}")
print(f"Sortino Ratio: {res['sortino_ratio']}")
print(f"Maximum Drawdown (MDD): {res['mdd']}")

abs sprd = 0.00000
GLD 180-day rolling avg = 0.00964
GC=F 180-day rolling avg = 0.01008
total trades = 215
winning trades = 173
win % = 0.80
Cumulative Returns: 0.8442004415230142
Sharpe Ratio: 0.8565774942217351
Sortino Ratio: 3.079384044837804
Maximum Drawdown (MDD): 0.013841738251418743
